# Explore CHAOS CT/MRI

This notebook uses the standard PyHealth flow: CHAOS dataset → `set_task()` → `SampleDataset` → `get_dataloader()`. The image is a DICOM series and the mask is a directory of PNG slices. Both are lazy: `set_task()` only records paths and metadata, and `show_chaos_samples()` can overlay the mask on the image when a sample is read. Data and caches default to `~/pyhealth-imaging-data` and `~/.cache/pyhealth-imaging` (`PYHEALTH_IMAGING_DATA` / `PYHEALTH_IMAGING_CACHE` override them); `download=True` fetches CHAOS automatically.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from pyhealth.datasets import CHAOSDataset, get_dataloader

# Override these with environment variables if your data/cache live elsewhere:
#   PYHEALTH_IMAGING_DATA
#   PYHEALTH_IMAGING_CACHE
DATA_ROOT = Path(
    os.environ.get("PYHEALTH_IMAGING_DATA", "~/pyhealth-imaging-data")
).expanduser()
CACHE_DIR = Path(
    os.environ.get("PYHEALTH_IMAGING_CACHE", "~/.cache/pyhealth-imaging")
).expanduser()
CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("TORCH_HOME", str(CACHE_DIR / "torch"))
print("Data directory:", DATA_ROOT)
print("Cache directory:", CACHE_DIR)

# Processed caches and generated metadata are stored here, outside the repo.
METADATA_DIR = CACHE_DIR / "metadata"


dataset = CHAOSDataset(
    root=DATA_ROOT / "CHAOS",
    cache_dir=CACHE_DIR,
    metadata_dir=str(METADATA_DIR / "chaos"),
    download=True,
)
dataset.stats()


def _display_gray(ax, image, title):
    """Show a CT/MR slice; MRI air/noise voxels are masked."""
    image = (
        image.detach().cpu().numpy()
        if hasattr(image, "detach")
        else np.asarray(image)
    )
    if float((image == 0).mean()) > 0.05:
        foreground = image[image != 0]
        vmin, vmax = np.percentile(foreground, [1, 99])
        if vmax <= vmin:
            vmax = vmin + 1.0
        ax.set_facecolor("black")
        ax.imshow(
            np.ma.masked_where(image == 0, image),
            cmap="gray",
            vmin=vmin,
            vmax=vmax,
        )
    else:
        vmin, vmax = np.percentile(image, [1, 99])
        if vmax <= vmin:
            vmax = vmin + 1.0
        ax.imshow(image, cmap="gray", vmin=vmin, vmax=vmax)
    ax.set_title(title)


def show_chaos_samples(samples, indices, overlay_mask=False):
    rows = 1 if overlay_mask else 2
    fig, axes = plt.subplots(
        rows, len(indices), figsize=(4 * len(indices), 4 * rows), squeeze=False
    )
    for col, index in enumerate(indices):
        sample = samples[index]
        image_vol = samples.input_processors["image"].materialize(sample["image"])
        mask_vol = samples.output_processors["mask"].materialize(sample["mask"])
        slice_idx = image_vol.shape[1] // 2
        image = image_vol[0, slice_idx].cpu()
        mask = mask_vol[0, slice_idx].cpu()
        title = (
            f"{sample['record_id']}\n"
            f"{sample.get('modality')}/{sample.get('sequence')}/{sample.get('phase')}"
        )
        if overlay_mask:
            _display_gray(axes[0, col], image, title)
            axes[0, col].imshow(
                np.ma.masked_where(mask == 0, mask), cmap="autumn", alpha=0.35
            )
        else:
            _display_gray(axes[0, col], image, title)
            axes[1, col].imshow(mask, cmap="gray")
            axes[1, col].set_title("mask")
        for row in range(rows):
            axes[row, col].axis("off")
    plt.show()


In [ ]:
patient_ids = dataset.unique_patient_ids
ct_id = next(pid for pid in patient_ids if pid.startswith("ct_"))
mr_id = next(pid for pid in patient_ids if pid.startswith("mr_"))
print("CT:", ct_id)
print("MR:", mr_id)
dataset.get_patient(ct_id).get_events()[0].attr_dict


In [ ]:
task = dataset.default_task
print(task.task_name)
print(task.input_schema)
print(task.output_schema)


In [ ]:
samples = dataset.set_task(task, num_workers=1)
print("samples:", len(samples))
print("input schema:", samples.input_schema)
print("output schema:", samples.output_schema)


In [ ]:
sample = samples[0]
image = samples.input_processors["image"].materialize(sample["image"])
mask = samples.output_processors["mask"].materialize(sample["mask"])
print(sample.keys())
print("image:", image.shape, image.dtype)
print("mask:", mask.shape, mask.dtype)
print("mask labels:", sorted(mask.unique().tolist()))


In [ ]:
loader = get_dataloader(samples, batch_size=1)
batch = next(iter(loader))
print("batch image:", batch["image"].shape)
print("batch mask:", batch["mask"].shape)


In [ ]:
record_ids = [samples[i]["record_id"] for i in range(len(samples))]

def find_index(token):
    return next(i for i, record_id in enumerate(record_ids) if token in record_id)

indices = []
for token in ["ct_", "_t1dual_inphase", "_t2spir"]:
    try:
        indices.append(find_index(token))
    except StopIteration:
        pass

show_chaos_samples(samples, indices, overlay_mask=True)


## Official CHAOS tasks and 3D segmentation training

Official task definitions: [https://chaos.grand-challenge.org/](https://chaos.grand-challenge.org/) and [https://chaos.grand-challenge.org/Combined_Healthy_Abdominal_Organ_Segmentation/](https://chaos.grand-challenge.org/Combined_Healthy_Abdominal_Organ_Segmentation/).

CHAOS defines five official tasks. In PyHealth each one is a **separate task class**, because the allowed modalities and label sets differ:

| Official task | PyHealth task class | Modalities | Organs | Classes |
|---|---|---|---|---|
| 1 | `CHAOSLiverSegmentationCTMRI` | CT + MRI | liver | 2 |
| 2 | `CHAOSLiverSegmentationCTonly` | CT | liver | 2 |
| 3 | `CHAOSLiverSegmentationMRIonly` | MRI | liver | 2 |
| 4 | `CHAOSAbdominalSegmentationCTMRI` | CT + MRI | liver, spleen, both kidneys (CT annotates liver only) | 5 |
| 5 | `CHAOSAbdominalSegmentationMRIonly` | MRI | liver, spleen, both kidneys | 5 |

This notebook trains **`CHAOSLiverSegmentationCTMRI`** by default (official task 1, cross-modality liver). Change `CHAOS_TASK` to another class for a different official task. The exploration cells above still use the dataset default, `CHAOSAbdominalSegmentationCTMRI`, so they can display all labels.

**How CT and MRI are fed:** this task does **not** pair CT and MR volumes. Each labelled series becomes one sample, `DicomSeriesProcessor` produces a single-channel 3D volume, and CT/MR samples are mixed in the same DataLoader. Normalization is selected per series (`ct` vs `mri`). The model is one `UNet3D` with `in_channels=1`; there is no modality embedding and no channel concatenation. This is the official cross-modality setting, not paired multimodal fusion.

For **task 4**, CT annotates the liver only while MRI annotates four organs. CT samples therefore carry `class_mask=[1, 1, 0, 0, 0]` (background + liver) and MR samples carry `class_mask=[1, 1, 1, 1, 1]`. The loss ignores unannotated spleen/kidney classes on CT instead of treating them as background, and metrics are filtered the same way.

CHAOS is volumetric and the official evaluation is volumetric, so training uses `UNet3D` on 3D patches. 2D-patch training is not used here; the notebook trains on 3D patches.

**MR display:** `normalize="mri"` z-scores the foreground and sets air/noise voxels to zero. The display helper masks zeros with a black background, so the plot shows anatomy instead of a noise halo.

**IO note:** each 3D training item reads and resamples one DICOM series. That decode/resample step is the main CPU/IO cost, and the GPU can be under-used while volumes are prepared. Precomputing a resampled slice cache or switching to a slice-level dataset would reduce it, but is not enabled here.


In [ ]:
import numpy as np
import torch
from collections import Counter

from pyhealth.datasets import PatientSplit, PatchDataset, get_dataloader
from pyhealth.metrics import per_class_segmentation_metrics
from pyhealth.models import UNet3D, evaluate_segmentation
from pyhealth.sampler import PatchSampler
from pyhealth.tasks import (
    CHAOSAbdominalSegmentationCTMRI,
    CHAOSAbdominalSegmentationMRIonly,
    CHAOSLiverSegmentationCTMRI,
    CHAOSLiverSegmentationCTonly,
    CHAOSLiverSegmentationMRIonly,
)
from pyhealth.trainer import Trainer

torch.backends.cudnn.benchmark = True

# Five official CHAOS tasks, one class per task.
CHAOS_TASKS = {
    "1_liver_ct_mri": CHAOSLiverSegmentationCTMRI,
    "2_liver_ct": CHAOSLiverSegmentationCTonly,
    "3_liver_mri": CHAOSLiverSegmentationMRIonly,
    "4_abdominal_ct_mri": CHAOSAbdominalSegmentationCTMRI,
    "5_abdominal_mri": CHAOSAbdominalSegmentationMRIonly,
}
CHAOS_TASK = CHAOSLiverSegmentationCTMRI
TARGET_SPACING = (2.0, 2.0, 2.0)
PATCH_SIZE = (64, 64, 64)
EPOCHS = 30
PATIENCE = 8
NUM_WORKERS = 4
PIN_MEMORY = torch.cuda.is_available()
MAX_TRAIN_CASES = None  # set an integer for a quicker run
MAX_VAL_CASES = None

device = "cuda" if torch.cuda.is_available() else "cpu"

task = CHAOS_TASK(target_spacing=TARGET_SPACING)
print("CHAOS task:", type(task).__name__)
print("official task:", task.official_task)
print("modalities:", task.modalities)
print("classes:", task.class_names)
print("class mask by modality:", task.class_mask_by_modality)

modality_counts = Counter()
for patient_id in dataset.unique_patient_ids:
    for event in dataset.get_patient(patient_id).get_events(event_type="series"):
        if getattr(event, "mask_dir", None) and event.modality in task.modalities:
            modality_counts[(event.modality, event.sequence, event.phase)] += 1
print("Training series by modality/sequence/phase:", modality_counts)

train_ds, val_ds = dataset.set_task(
    task,
    split=PatientSplit(ratios=(0.8, 0.2), seed=42),
    num_workers=1,
)
if MAX_TRAIN_CASES is not None and MAX_TRAIN_CASES < len(train_ds):
    rng = np.random.default_rng(42)
    train_ds = train_ds.subset(
        rng.choice(len(train_ds), MAX_TRAIN_CASES, replace=False)
    )
if MAX_VAL_CASES is not None and MAX_VAL_CASES < len(val_ds):
    rng = np.random.default_rng(43)
    val_ds = val_ds.subset(
        rng.choice(len(val_ds), MAX_VAL_CASES, replace=False)
    )

model = UNet3D(
    train_ds,
    in_channels=1,
    num_classes=task.num_classes,
)

train_patches = PatchDataset(
    train_ds,
    PatchSampler(PATCH_SIZE, foreground_prob=0.75),
    mode="train",
    num_patches=1,
)
val_patches = PatchDataset(
    val_ds,
    # Foreground patches measure segmentation quality; the case-level metric
    # below is the unbiased volumetric result.
    PatchSampler(PATCH_SIZE, foreground_prob=1.0),
    mode="train",
    num_patches=1,
)
loader_kwargs = dict(num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
if NUM_WORKERS > 0:
    loader_kwargs.update(persistent_workers=True, prefetch_factor=2)
train_loader = get_dataloader(
    train_patches, batch_size=1, shuffle=True, **loader_kwargs
)
val_loader = get_dataloader(
    val_patches, batch_size=1, **loader_kwargs
)

print("parameters (M):", sum(p.numel() for p in model.parameters()) / 1e6)
trainer = Trainer(model, device=device, metrics=["dice", "iou"])
trainer.train(
    train_loader,
    val_dataloader=val_loader,
    epochs=EPOCHS,
    optimizer_params={"lr": 3e-4},
    weight_decay=1e-4,
    max_grad_norm=1.0,
    monitor="dice",
    monitor_criterion="max",
    patience=PATIENCE,
)

print("patch-level validation:", trainer.evaluate(val_loader))

y_true, y_prob, _ = trainer.inference(val_loader)
per_class = per_class_segmentation_metrics(
    y_true,
    y_prob,
    mode=model.mode,
    num_classes=task.num_classes,
    ignore_index=getattr(model, "segmentation_ignore_index", None),
)
print("per-class validation Dice (background excluded):")
for class_index, class_name in enumerate(task.class_names, start=1):
    value = per_class.get(f"dice_class_{class_index}", float("nan"))
    print(f"  {class_index} {class_name}: {value:.4f}")

if torch.cuda.is_available():
    torch.cuda.empty_cache()
print(
    "case-level validation:",
    evaluate_segmentation(
        model,
        val_ds,
        patch_size=PATCH_SIZE,
        overlap=0.5,
        batch_size=1,
    ),
)

run_dir = CACHE_DIR / "runs"
run_dir.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), run_dir / "chaos_unet3d.ckpt")